# Debug Notebook: GPU Optimization Benchmark

This notebook benchmarks speculative decoding with GPU optimization switches OFF vs ON,
then produces an automatic delta table for:
- draft time (`draft_elapsed_s`)
- verify time (`verify_elapsed_s`)
- end-to-end latency (`latency_s`)

Outputs are saved to `results/gpu_opt_benchmark_runs.csv` and `results/gpu_opt_benchmark_delta.csv`.

In [ ]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'torch is required for this notebook. Select a Python environment with PyTorch installed.'
    ) from exc

# Resolve project root robustly (works from notebook root or subfolders).
search_starts = []
if '__file__' in globals():
    search_starts.append(Path(__file__).resolve().parent)
search_starts.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_starts:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'speculative.py').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/speculative.py')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior for reproducibility in debug sessions.
os.environ.setdefault('SPECDEC_HF_OFFLINE_FIRST', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for this benchmark notebook.')

import speculative as spec_mod
from config import DRAFT_MODELS, DRAFT_QUANT, TARGET_MODEL_ID, TARGET_QUANT, REGIMES
from speculative import load_model_on_device
from utils import set_seed

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
manifests_dir = project_root / 'manifests'

print(f'Project root: {project_root}')
print(f'CUDA device count: {torch.cuda.device_count()}')

In [ ]:
# Benchmark controls
DEVICE = 'cuda:0'
DRAFT_LABEL = '0.5B'
REGIME_NAME = 'deterministic'   # set to 'stochastic' if needed
K = 8
MAX_NEW_TOKENS = 64
N_PER_TASK = 4
BASE_SEED = 2026
WARMUP_NEW_TOKENS = 16

# Conditions: OFF (reference) vs ON (optimized)
CONDITIONS = [
    {
        'condition': 'gpu_opt_off',
        'GPU_USE_SEPARATE_STREAMS': False,
        'GPU_PREALLOCATE_STEP_BUFFERS': False,
        'GPU_USE_STABLE_STEP_SHAPES': False,
        'GPU_TRY_CUDA_GRAPHS': False,
    },
    {
        'condition': 'gpu_opt_on',
        'GPU_USE_SEPARATE_STREAMS': True,
        'GPU_PREALLOCATE_STEP_BUFFERS': True,
        'GPU_USE_STABLE_STEP_SHAPES': True,
        'GPU_TRY_CUDA_GRAPHS': False,
    },
]

print('Benchmark config:')
print(f'  device={DEVICE}, target_quant={TARGET_QUANT}, draft_quant={DRAFT_QUANT}')
print(f'  draft={DRAFT_LABEL}, regime={REGIME_NAME}, k={K}, max_new_tokens={MAX_NEW_TOKENS}')
print(f'  n_per_task={N_PER_TASK}, warmup_new_tokens={WARMUP_NEW_TOKENS}')

In [ ]:
def load_prompt_rows(n_per_task: int = 4) -> list[dict]:
    rows = []
    for task in ['gsm8k', 'mmlu', 'cnndm']:
        path = manifests_dir / f'{task}_data.json'
        if not path.exists():
            continue
        with open(path, 'r') as f:
            data = json.load(f)
        for item in data[:n_per_task]:
            prompt = item.get('prompt')
            if prompt:
                rows.append({
                    'task': task,
                    'sample_id': item.get('sample_id', ''),
                    'prompt': prompt,
                })
    if not rows:
        raise RuntimeError('No prompts found in manifests/*_data.json')
    return rows

prompt_rows = load_prompt_rows(N_PER_TASK)
print(f'Loaded prompts: {len(prompt_rows)}')
pd.DataFrame(prompt_rows)[['task', 'sample_id']].head(12)

In [ ]:
# Load models once and reuse across both conditions.
target_model, target_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    device=DEVICE,
    quant_mode=TARGET_QUANT,
)
draft_model, _ = load_model_on_device(
    DRAFT_MODELS[DRAFT_LABEL],
    device=DEVICE,
    quant_mode=DRAFT_QUANT,
)

print('Models loaded.')
print(f'  target: {TARGET_MODEL_ID}')
print(f'  draft : {DRAFT_MODELS[DRAFT_LABEL]}')

In [ ]:
def apply_condition_flags(cfg: dict) -> None:
    spec_mod.GPU_USE_SEPARATE_STREAMS = bool(cfg['GPU_USE_SEPARATE_STREAMS'])
    spec_mod.GPU_PREALLOCATE_STEP_BUFFERS = bool(cfg['GPU_PREALLOCATE_STEP_BUFFERS'])
    spec_mod.GPU_USE_STABLE_STEP_SHAPES = bool(cfg['GPU_USE_STABLE_STEP_SHAPES'])
    spec_mod.GPU_TRY_CUDA_GRAPHS = bool(cfg['GPU_TRY_CUDA_GRAPHS'])

def run_condition(condition_cfg: dict, prompts: list[dict], base_seed: int = 0) -> pd.DataFrame:
    condition_name = condition_cfg['condition']
    apply_condition_flags(condition_cfg)
    regime = REGIMES[REGIME_NAME]

    # Warm-up (excluded from metrics).
    warm_prompt = prompts[0]['prompt']
    set_seed(base_seed)
    _ = spec_mod.speculative_decode_sample(
        target_model,
        draft_model,
        target_tokenizer,
        warm_prompt,
        max_new_tokens=min(WARMUP_NEW_TOKENS, MAX_NEW_TOKENS),
        k=K,
        temperature=regime.temperature,
        top_p=regime.top_p,
        return_timing_breakdown=True,
    )

    records = []
    for i, row in enumerate(prompts):
        seed = base_seed + i
        set_seed(seed)
        out = spec_mod.speculative_decode_sample(
            target_model,
            draft_model,
            target_tokenizer,
            row['prompt'],
            max_new_tokens=MAX_NEW_TOKENS,
            k=K,
            temperature=regime.temperature,
            top_p=regime.top_p,
            return_timing_breakdown=True,
        )

        records.append({
            'condition': condition_name,
            'task': row['task'],
            'sample_id': row['sample_id'],
            'seed': seed,
            'k': K,
            'max_new_tokens': MAX_NEW_TOKENS,
            'latency_s': float(out.get('latency_s', 0.0)),
            'draft_elapsed_s': float(out.get('draft_elapsed_s', 0.0)),
            'verify_elapsed_s': float(out.get('verify_elapsed_s', 0.0)),
            'ttft_ms': float(out.get('ttft_ms', 0.0)),
            'num_tokens': int(out.get('num_tokens', 0)),
            'alpha': float(out.get('alpha', 0.0)),
            'B_eff': float(out.get('B_eff', 0.0)),
            'gpu_stream_pipeline': bool(out.get('gpu_stream_pipeline', False)),
            'gpu_stable_step_shapes': bool(out.get('gpu_stable_step_shapes', False)),
            'gpu_graph_capture': out.get('gpu_graph_capture', 'unknown'),
        })

    return pd.DataFrame(records)

In [ ]:
all_runs = []
for idx, cfg in enumerate(CONDITIONS):
    print(f"Running {cfg['condition']} ({idx + 1}/{len(CONDITIONS)})...")
    df_cond = run_condition(cfg, prompt_rows, base_seed=BASE_SEED)
    all_runs.append(df_cond)

df_runs = pd.concat(all_runs, ignore_index=True)
print(f'Benchmark rows: {len(df_runs)}')
df_runs.head()

In [ ]:
metrics = ['draft_elapsed_s', 'verify_elapsed_s', 'latency_s']
summary = df_runs.groupby('condition', as_index=False)[metrics].mean()

off = summary[summary['condition'] == 'gpu_opt_off'].iloc[0]
on = summary[summary['condition'] == 'gpu_opt_on'].iloc[0]

delta_rows = []
for m in metrics:
    off_v = float(off[m])
    on_v = float(on[m])
    abs_delta = on_v - off_v
    pct_delta = (abs_delta / off_v * 100.0) if off_v != 0 else float('nan')
    gain_pct = (-pct_delta) if pd.notna(pct_delta) else float('nan')
    delta_rows.append({
        'metric': m,
        'gpu_opt_off_mean': off_v,
        'gpu_opt_on_mean': on_v,
        'abs_delta_on_minus_off': abs_delta,
        'pct_delta_on_minus_off': pct_delta,
        'improvement_pct_lower_is_better': gain_pct,
        'ratio_on_over_off': (on_v / off_v) if off_v != 0 else float('nan'),
    })

delta_table = pd.DataFrame(delta_rows)
delta_table

In [ ]:
task_delta = (
    df_runs.groupby(['task', 'condition'])[['draft_elapsed_s', 'verify_elapsed_s', 'latency_s']]
    .mean()
    .reset_index()
)

task_pivot = task_delta.pivot(index='task', columns='condition', values=['draft_elapsed_s', 'verify_elapsed_s', 'latency_s'])

task_rows = []
for task in task_pivot.index:
    row = {'task': task}
    for metric in ['draft_elapsed_s', 'verify_elapsed_s', 'latency_s']:
        off_v = float(task_pivot.loc[task, (metric, 'gpu_opt_off')])
        on_v = float(task_pivot.loc[task, (metric, 'gpu_opt_on')])
        row[f'{metric}_off'] = off_v
        row[f'{metric}_on'] = on_v
        row[f'{metric}_delta_pct'] = ((on_v - off_v) / off_v * 100.0) if off_v != 0 else float('nan')
    task_rows.append(row)

task_delta_table = pd.DataFrame(task_rows)
task_delta_table

In [ ]:
runs_csv = results_dir / 'gpu_opt_benchmark_runs.csv'
delta_csv = results_dir / 'gpu_opt_benchmark_delta.csv'
task_delta_csv = results_dir / 'gpu_opt_benchmark_task_delta.csv'

df_runs.to_csv(runs_csv, index=False)
delta_table.to_csv(delta_csv, index=False)
task_delta_table.to_csv(task_delta_csv, index=False)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
df_runs.to_csv(results_dir / f'gpu_opt_benchmark_runs_{stamp}.csv', index=False)
delta_table.to_csv(results_dir / f'gpu_opt_benchmark_delta_{stamp}.csv', index=False)

print('Saved benchmark outputs:')
print(f'  {runs_csv}')
print(f'  {delta_csv}')
print(f'  {task_delta_csv}')

delta_table